In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df[["time", "X", "Y", "ndvi"]]

In [ ]:
import xee
help(xee.EarthEngineBackendEntrypoint)

In [2]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()


Successfully saved authorization token.


In [1]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

#US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
#WSDemo_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
#Park_Lane_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/park_lane_tahoe")
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2015-05-01', '2015-09-30').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])
#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2015-01-01', '2015-05-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=Park_Lane_Boundaries.geometry(), crs='EPSG:3310', scale=30)

In [ ]:
print(ds)

### Using R code instead of Python

You can also run your function via R code on the Docker workers! Instead of a Python function, you define your function as an R script string.

In [ ]:
r_code = """\
compute_ndvi_r <- function(df) {
    df$ndvi <- (df$SR_B5 - df$SR_B4) / (df$SR_B5 + df$SR_B4)
    return(df[, c("time", "X", "Y", "ndvi")])
}
"""

df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}
from robustraster import run

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    max_pixels_per_tile = 1_000_000,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "is_r_function": True,
        "r_function_code": r_code,
        "r_function_name": "compute_ndvi_r",
    },
    function_tuning_config={
        "tune_function": False,
        "chunks": chunks,
        "max_iterations": None,
        "output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles_R_Docker",
        "vrt": True,
        "report": True
    },
    dask_mode="custom",
    dask_config={
        "n_workers": 6,
        "threads_per_worker": 1,
        "memory_limit": "3g",
    },
    dask_use_docker=True,
    dask_docker_image="adrianomdocker/r042"
)

[robustraster] Found EE credentials at: C:\Users\Adriano Matos/.config/earthengine/credentials
[robustraster] Mounting EE credentials to /root/.config/earthengine/credentials
[robustraster] Mounting output: C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\demos\Plumas_NDVI_Tiles_R_Docker -> /robustraster_output
Dask Scheduler started: dask-cb0e53-scheduler (dcf63b5cbec0)
Dask Worker 1 started: dask-cb0e53-worker-1 (4131e6a6620f) mapped to port 62483
Dask Worker 2 started: dask-cb0e53-worker-2 (3de70e31fbc6) mapped to port 62484
Dask Worker 3 started: dask-cb0e53-worker-3 (b33ab09ee088) mapped to port 62485
Dask Worker 4 started: dask-cb0e53-worker-4 (9bbafe36b901) mapped to port 62486
Dask Worker 5 started: dask-cb0e53-worker-5 (45ed9b6a8989) mapped to port 62487
Dask Worker 6 started: dask-cb0e53-worker-6 (8530b38abd7e) mapped to port 62488


c:\anaconda3\envs\rr042_rpy2\lib\site-packages\distributed\client.py:1617: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.6  | 2.0.1     | 2.0.1   |
| tornado | 6.5.4  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Docker Dask cluster started: <Client: 'tcp://172.20.0.2:8786' processes=4 threads=4, memory=11.18 GiB>
[robustraster] Dask workers authenticated to Earth Engine.


[robustraster] Running user function...
Exported: /robustraster_output/x-22980_-3660_y213009_255969__time_20150501T183819.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-22980_-3660_y151569_212979__time_20150501T183819.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-145860_-84450_y213009_255969__time_20150501T183819.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-84420_-23010_y213009_255969__time_20150501T183819.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-145860_-84450_y151569_212979__time_20150501T183819.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-145860_-84450_y151569_212979__time_20150501T183843.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-84420_-23010_y151569_212979__time_20150501T183819.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-22980_-3660_y151569_212979__time_20150609T184431.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x

ConnectionError: None: Max retries exceeded with url: /token (Caused by ConnectTimeoutError(<HTTPSConnection(host='oauth2.googleapis.com', port=443) at 0x2502541b0d0>, 'Connection to oauth2.googleapis.com timed out. (connect timeout=None)'))